# 9 Working with Polar Orbiting Sensor Datasets

## 9.1 Visible Infrared Imaging Radiometer Suite (VIIRS)


## 9.1.1 Importing and Plotting

In [ ]:
import datetime
# import glob
from pathlib import Path
# import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from cartopy import crs as ccrs
import pvlib

In [ ]:
notebook_dir = Path.cwd()
fpath = notebook_dir / "data"

In [ ]:
# Optional: Uniformly make font sizes larger and set default figure size
plt.rcParams.update({"font.size": 16, 
    "figure.figsize": [12, 12],
    "font.sans-serif": ["Segoe UI", "DejaVu Sans", "sans-serif"]
    })

CBAR_FRAC = 0.05
CBAR_PAD = 0.025

In [ ]:
# Using IR bands
filename1 = fpath / "viirs" / "VJ202MOD.A2024254.0624.002.2024254125517.nc"
f1 = xr.open_datatree(filename1, mask_and_scale=False, engine="h5netcdf", decode_timedelta=True)
print("List of groups:", list(f1.groups))

In [ ]:
group1 = f1["observation_data"]
print("Datasets in group:", list(group1))

In [ ]:
rad_viirs_mband = group1["M12"]
rad_viirs_mband.attrs

In [ ]:
scale_factor_viirs_mband = rad_viirs_mband.attrs["scale_factor"].item()
add_offset_viirs_mband = rad_viirs_mband.attrs["add_offset"].item()
rad_unpacked_viirs_mband = \
  rad_viirs_mband[:,:] * scale_factor_viirs_mband \
  + add_offset_viirs_mband

In [ ]:
plt.figure()
ax = plt.subplot()
img = ax.imshow(rad_unpacked_viirs_mband, vmin=0, vmax=0.5)
plt.colorbar(img, fraction=CBAR_FRAC, pad=CBAR_PAD, label="Radiance (W m-2 sr-1 um-1)")
ax.set(frame_on=True, xticks=[], yticks=[])
plt.show()

In [ ]:

filename2 = fpath / "viirs" / "VJ203MOD.A2024254.0624.002.2024254124710.nc"

f2 = xr.open_datatree(filename2, mask_and_scale=False, \
  engine="h5netcdf", decode_timedelta=True)
print("Groups:", list(f2.groups))
data_group2 = f2["geolocation_data"]
print("Variables in geolocation_data:", list(data_group2))

In [ ]:
lats_viirs_mband = data_group2["latitude"]
lons_viirs_mband = data_group2["longitude"]

In [ ]:
fig = plt.figure(figsize=[10,5])
ax = plt.subplot(projection=ccrs.PlateCarree())
ax.coastlines("50m")
img = ax.pcolormesh(lons_viirs_mband, lats_viirs_mband, rad_unpacked_viirs_mband, vmin=0, vmax=0.5)
plt.colorbar(img, fraction=CBAR_FRAC, pad=CBAR_PAD)
plt.show()

## 9.1.2 Bowtie effects

In [ ]:
# Extract bowtie deletion flag - it"s a single value
bowtie_flag = rad_viirs_mband.attrs["flag_values"][1]

In [ ]:
mask = (rad_viirs_mband[:,:] == bowtie_flag)

In [ ]:
# xarray doesn"t support 2d boolean subsetting at the time of writing
# need to conver from ndarray to nparray
rad_unpacked_viirs_mband = np.array(rad_unpacked_viirs_mband)
rad_unpacked_viirs_mband[mask] = np.nan

In [ ]:
vza = data_group2["sensor_zenith"]/100
vaa = data_group2["sensor_azimuth"]/100

In [ ]:
# Constained layout prevents text overlap
fig, ax = plt.subplots(3, 1, constrained_layout=True)

img = ax[0].imshow(rad_unpacked_viirs_mband[0:300, :], vmin=0.1, vmax=0.5)
fig.colorbar(img, orientation="horizontal")
ax[0].set_title("Radiance")

img = ax[1].imshow(vza[0:300, :], cmap="jet")
fig.colorbar(img, orientation="horizontal", label="Degrees")
ax[1].set_title("View Zenith Angle")

img = ax[2].imshow(vaa[0:300, :], cmap="jet")
fig.colorbar(img, orientation="horizontal", label="Degrees")
ax[2].set_title("View Azimuth Angle")

for ax0 in ax:
  ax0.set(frame_on=True, xticks=[], yticks=[])

plt.show()

In [ ]:
import scipy.ndimage as snd

In [ ]:
rad_unpacked_viirs_mband_copy = rad_unpacked_viirs_mband.copy()
rad_unpacked_viirs_mband_copy[mask] = 0

# Left side
left_boundary = np.where( (vza >= 31.72) & (vaa >= 0) )[1].max()
rad_filled_lhs = snd.maximum_filter1d(rad_unpacked_viirs_mband_copy[:, :left_boundary], size=10, axis=0, mode="nearest")

# Right side
right_boundary = np.where( (vza >= 31.72) & (vaa <= 0) )[1].min()
rad_filled_rhs = snd.maximum_filter1d(rad_unpacked_viirs_mband_copy[:,right_boundary:], size=10, axis=0, mode="nearest")

# Center unchanged
rad_filled_center = rad_unpacked_viirs_mband_copy[:, left_boundary:right_boundary]

In [ ]:
rad_filled_viirs_mband = np.concatenate([rad_filled_lhs, \
  rad_filled_center, rad_filled_rhs], axis=1)

In [ ]:
rad_combined_viirs_mband = np.where(mask, rad_filled_viirs_mband, \
  rad_unpacked_viirs_mband)

In [ ]:
fig = plt.figure(figsize=[10,5])
ax = plt.subplot(projection=ccrs.PlateCarree())
ax.coastlines("50m")
img = ax.pcolormesh(lons_viirs_mband, lats_viirs_mband, \
    rad_combined_viirs_mband, vmin=0, vmax=0.5)
plt.colorbar(img, fraction=CBAR_FRAC, pad=CBAR_PAD)
plt.show()

In [ ]:
# Used for Fig 9.6
fig, ax = plt.subplots(1,3, constrained_layout=True)

img = ax[0].imshow(rad_unpacked_viirs_mband[0:200, 0:200], vmin=0, vmax=0.2)
ax[0].set_title("(a) Uncorrected")

img = ax[1].imshow(rad_filled_viirs_mband[0:200, 0:200], vmin=0, vmax=0.2)
ax[1].set_title("(b) Filtered")


img = ax[2].imshow(rad_combined_viirs_mband[0:200, 0:200], vmin=0, vmax=0.2)
fig.colorbar(img, orientation="vertical", fraction=0.05, pad=0.05)
ax[2].set_title("(c) Corrected")

for ax0 in ax.ravel():
  ax0.set(frame_on=True, xticks=[], yticks=[])

plt.show()

In [ ]:
f1.close()
f2.close()

## 9.2 Atmospheric Correction

In [ ]:
# Using imagery now - applying mask and scale this time
filename3 = fpath / "viirs" / "VJ102IMG.A2024254.2018.021.2024255022451.nc"
f1 = xr.open_datatree(filename3, mask_and_scale=True, \
  engine="h5netcdf", decode_timedelta=True)
group1 = f1["observation_data"]

filename4 = fpath / "viirs" / "VJ103IMG.A2024254.2018.021.2024255015215.nc"
f2 = xr.open_datatree(filename4, mask_and_scale=True, \
  engine="h5netcdf", decode_timedelta=True)
data_group2 = f2["geolocation_data"]

In [ ]:
print("Datasets in group:", list(group1))

In [ ]:
# Subset the data for faster processing
def create_sample(var, start_x=4200, start_y=3300, delta=500, step=1):
  end_x = start_x + delta
  end_y = start_y + delta
  return var[start_x:end_x, start_y:end_y][::step, ::step]

In [ ]:
rad_viirs_iband = create_sample(group1["I01"])
lats_viirs_iband = create_sample(data_group2["latitude"])
lons_viirs_iband = create_sample(data_group2["longitude"])

In [ ]:
sza = create_sample(data_group2["solar_zenith"])
saa = create_sample(data_group2["solar_azimuth"])
vza = create_sample(data_group2["sensor_zenith"])
vaa = create_sample(data_group2["sensor_azimuth"])

In [ ]:
fig, ax = plt.subplots(2,2, figsize=[10,10])
label = "Degrees (°)"

img = ax[0,0].imshow(sza, cmap="jet")
plt.colorbar(img, orientation="horizontal", label=label)
ax[0,0].set_title("Solar Zenith Angle")

img = ax[0,1].imshow(saa, cmap="jet")
plt.colorbar(img, orientation="horizontal", label=label)
ax[0,1].set_title("Solar Azimuth Angle")

img = ax[1,0].imshow(vza, cmap="jet")
plt.colorbar(img, orientation="horizontal", label=label)
ax[1,0].set_title("View Zenith Angle")

img = ax[1,1].imshow(vaa, cmap="jet")
plt.colorbar(img, orientation="horizontal", label=label)
ax[1,1].set_title("View Azimuth Angle")

for ax0 in ax.ravel():
    ax0.set(frame_on=True, xticks=[], yticks=[])

plt.show()

### 9.2.1 Unit Conversions

Anderson, K., Hansen, C., Holmgren, W., Jensen, A., Mikofski, M., and Driesse, A. “pvlib python: 2023 project update.” Journal of Open Source Software, 8(92), 5994, (2023). DOI: 10.21105/joss.05994.

In [ ]:
am15 = pvlib.spectrum.get_reference_spectra(standard="ASTM G173-03")

In [ ]:
solar_irradiance = pvlib.spectrum.get_reference_spectra(standard="ASTM G173-03")
toa_irradiance_um = solar_irradiance["extraterrestrial"]
wavelengths_um = solar_irradiance.index

In [ ]:
plt.figure(figsize=[10,5])
plt.plot(wavelengths_um, toa_irradiance_um)
plt.xlabel("Wavelength (nm)")
plt.ylabel("Irradiance (W m-2 nm-1)")
plt.title("Solar Spectral Irradiance (Top of Atmosphere)")
plt.xlim(250, 1300)
plt.show()

In [ ]:
def radiance_to_reflectance(L, Es, theta_s_deg, distance_au):
    theta_s_rad = np.deg2rad(theta_s_deg)
    reflectance = (np.pi * L) / (Es * np.cos(theta_s_rad) / distance_au**2)
    return reflectance

In [ ]:
obs_time = f1.time_coverage_start
obs_time

In [ ]:
fmt = "%Y-%m-%dT%H:%M:%S.%fZ"
obs_time_dt = datetime.datetime.strptime(obs_time, fmt)
obs_time_dt


In [ ]:
distance_au = pvlib.solarposition.nrel_earthsun_distance(\
  obs_time_dt, how="numpy").item()
print(distance_au)

In [ ]:
# Calculate solar irradiance in (W·m⁻²·nm⁻¹)
solar_irradiance_640nm = solar_irradiance[solar_irradiance.index == 640.0].extraterrestrial.item()
print(solar_irradiance_640nm)

In [ ]:
# Convert radiance to reflectance
ref_viirs_iband = radiance_to_reflectance(rad_viirs_iband, \
    solar_irradiance_640nm, sza, distance_au)
print(solar_irradiance_640nm)


In [ ]:
counts_orig_refl, bins_orig_refl = np.histogram(ref_viirs_iband, density=True, bins=100)
counts_orig_rad, bins_orig_rad = np.histogram(rad_viirs_iband, density=True, bins=100)

In [ ]:
plt.figure(figsize=[10,5])
plt.plot(bins_orig_refl[:-1], counts_orig_refl, label="Reflectance (W·m⁻²·sr⁻¹·μm⁻¹)")
plt.plot(bins_orig_rad[:-1], counts_orig_rad, label="Radiance (unitless)")
plt.title("Reflectance vs Radiance")
plt.ylabel("Probability Density")
plt.xlim(0, 1.0)
plt.legend()
plt.show()

### 9.2.2 Dark Object Subtraction

In [ ]:
# Dark Object Subtraction (DOS)
dark_pixel_viirs_iband = np.min(group1["I01"])
dark_pixel_viirs_iband.item()

In [ ]:
rad_dos_viirs_iband = rad_viirs_iband - dark_pixel_viirs_iband

In [ ]:
def create_density(sample_orig, sample_corr, label_orig="Original", label_corr="Corrected", units="Radiance", xmax=0.3, bins=(100,100)):
  if units == "Radiance":
    xlabel = units+" (W m-2 sr-1 um-1)"
    range=None
  elif units == "Reflectance":
    xlabel = units
    range=(0,1)
  else:
    plt.xlabel("")
    range=None

  # Title
  title = label_orig + " " + units + " vs. "+label_corr + " " + units


  # Compare density
  density_orig, bins_orig = np.histogram(sample_orig, \
    bins=bins[0], density=True, range=range)
  density_corr, bins_corr = np.histogram(sample_corr, \
    bins=bins[1], density=True, range=range)

  plt.figure(figsize=[10,5])
  plt.plot(bins_orig[:-1], density_orig, label=label_orig)
  plt.plot(bins_corr[:-1], density_corr, label=label_corr)
  plt.title(title)
  plt.xlabel(xlabel)

  plt.ylabel("Probability Density")
  plt.xlim(0, xmax)
  plt.legend()
  plt.show()

In [ ]:
create_density(rad_viirs_iband, rad_dos_viirs_iband, label_corr="Dark Object Subtraction")

In [ ]:
# fig, ax = plt.subplots(1,2, figsize=[10,6])

# img = ax[0].imshow(rad_viirs_iband[300:400, 300:400], cmap="tab20c", vmin=0, vmax=0.25)
# # plt.colorbar(img, orientation="horizontal", label = "Radiance")
# ax[0].set_title("Original")

# img = ax[1].imshow(rad_dos_viirs_iband[300:400, 300:400], cmap="tab20c", vmin=0, vmax=0.25)
# ax[1].set_title("Dark Object Subtraction")

# fig.colorbar(img, ax=ax, orientation="horizontal", label = "Radiance", pad=0.1)

# plt.show()

### 9.2.2 Rayleigh Correction

In [ ]:
def rayleigh_single(xtau, xphi, xmuv, xmus):
  # Kronecker Terms (Eq. 9.4)
  kron = [1.0, 2.0, 2.0]

  # Cosine terms (Eq. 9.4 )
  phios = np.pi-xphi
  cosf = [1.0, np.cos(phios), np.cos(2.0*phios)]

  # Rayleigh phase function terms (Eq. 9.8-9.9)
  beta = 0.5
  gamma = 0.0279

  # Constant from Eqn. 9.10, used in Eqn. 9.7-9.9
  alpha = (1-gamma/(2-gamma))/(1+2*gamma/(2-gamma))

  # Eqs. 9.7-9.9
  P0 = 1.0 + (3*xmus**2-1.0)*(3*xmuv**2-1)*alpha/8.0
  P1 = -xmus*xmuv*np.sqrt(1-xmus**2) \
      * np.sqrt(1-xmuv**2) \
      * alpha*beta*1.5
  P2 = (1-xmus**2)*(1-xmuv**2)*alpha*beta*0.375

  # Eq. 9.6
  xitm = (1-np.exp(-xtau*(1/xmus+1/xmuv)))*xmus/(4*(xmus+xmuv))
  phase = [P0*xitm, P1*xitm, P2*xitm]

  # Single scattering component (Eq. 9.4)
  xray = kron[0]*phase[0]*cosf[0] \
    + kron[1]*phase[1]*cosf[1] \
    + kron[2]*phase[2]*cosf[2]

  return xray

In [ ]:
# Read rayleigh optical depth for standard atmosphere
tray = pd.read_csv(fpath / "txt" / "rayleigh_optical_depth.csv", sep=",")
xtau = tray[tray.wavelength_um == 0.64].Rayleigh_optical_depth.item()

In [ ]:
solar_zenith_rad = np.deg2rad(sza)
view_zenith_rad = np.deg2rad(vza)
solar_azimuth = np.deg2rad(saa)
view_azimuth = np.deg2rad(vaa)

relative_azimuth_rad = solar_azimuth - view_azimuth

In [ ]:
# Rayleigh scattering function
# xphi: azimuthal difference between sun and observation (xphi=0, in backscattering) and expressed in degree (0 to 360.)
# xmus: cosine of the sun zenith angle
# xmuv: cosine of the observation zenith angle
# xtau: raylegh optical depth

xphi = relative_azimuth_rad
xmus = np.cos(solar_zenith_rad)
xmuv = np.cos(view_zenith_rad)

In [ ]:
ray_radiance = rayleigh_single(xtau, xphi, xmuv, xmus)

In [ ]:
fig, ax = plt.subplots(1,1)
img = ax.imshow(ray_radiance, cmap="jet")
fig.colorbar(img, orientation="vertical", pad=0.1, fraction=0.05)
ax.set_title("Rayleigh Radiance")
ax.set(frame_on=True, xticks=[], yticks=[])
plt.show()

In [ ]:
create_density(rad_viirs_iband, rad_viirs_iband-ray_radiance, label_corr="Rayleigh-Corrected")

### 9.2.2 Radiative Transfer Models

Note: removing from chapter since 6S is not cross platform.

In [ ]:
# Using RTM
# from Py6S import *

In [ ]:
# spacing = 0.0125
# refl_range = np.arange(0, 1, spacing)

In [ ]:
# # Create a BRDF Look-Up Table (LUT)
# py6s_correction = np.zeros([len(refl_range)])

# s = SixS()

# # Input Wavelength = VIIRS Imagery Band 1
# s.wavelength = Wavelength(PredefinedWavelengths.VIIRS_BI1)

# # Altitudes
# s.altitudes.set_target_sea_level()
# s.altitudes.set_sensor_satellite_level()

# # Parameters
# s.aero_profile = AeroProfile.PredefinedType(AeroProfile.Continental)
# s.aot550 = 0.1
# s.atmos_profile = AtmosProfile.PredefinedType(AtmosProfile.MidlatitudeSummer)
# s.ground_reflectance = GroundReflectance.HomogeneousRoujean(0.037, 0.0, 0.133) # Pine forest, visible parameters

# # Geometry
# s.geometry = Geometry.User()
# s.geometry.solar_z = np.mean(solar_zenith_rad)
# s.geometry.view_z = np.mean(view_zenith_rad)
# s.geometry.solar_a = np.mean(solar_azimuth)
# s.geometry.view_a = np.mean(view_azimuth)
# s.geometry.latitude = np.mean(lats_viirs_iband)
# s.geometry.longitude = np.mean(lons_viirs_iband)

# # Datetime
# s.geometry.month = obs_time_dt.month
# s.geometry.day = obs_time_dt.day
# s.geometry.gmt_decimal_hour = obs_time_dt.hour + obs_time_dt.minute/60 + obs_time_dt.second/3600

# for i, rad_i in enumerate(refl_range):
#       # Input Reflectance
#       s.atmos_corr = AtmosCorr.AtmosCorrBRDFFromReflectance(rad_i)

#       s.run()

#       # Save corrected value to LUT
#       py6s_correction[i] = s.outputs.atmos_corrected_reflectance_brdf

In [ ]:
# Create dictionary with values
# lut = dict(zip(refl_range, py6s_correction))

In [ ]:
# Round reflectance to the nearest 0.0125
# reflectance_rounded = ref_viirs_iband.copy()
# reflectance_rounded = np.round(reflectance_rounded / spacing) * spacing

In [ ]:
# reflectance_corrected = np.copy(np.array(reflectance_rounded))

# # Loop through dictionary and replace original values
# for key, value in lut.items():
#     mask = reflectance_corrected == key
#     reflectance_corrected[mask] = value
# # Examine how many values replaced
# np.sum(reflectance_corrected != reflectance_rounded)/reflectance_rounded.size

In [ ]:
# reflectance_corrected = np.copy(np.array(reflectance_rounded))

# # Tolerance for matching spacing=0.0025/2
# tolerance = spacing/2

# # Loop through dictionary and apply replacements where close
# for key, value in lut.items():
#     mask = np.isclose(reflectance_rounded, key, atol=tolerance)
#     reflectance_corrected[mask] = value

# # Examine how many values replaced
# np.sum(reflectance_corrected != reflectance_rounded)/reflectance_rounded.size

In [ ]:
# np.sum(reflectance_corrected != reflectance_rounded)/reflectance_rounded.size

In [ ]:
# mask = np.isclose(reflectance_rounded, 0.1, atol=tolerance)
# print("Original Array:", np.array(ref_viirs_iband)[mask][0])
# print("Rounded Array:", np.array(reflectance_rounded)[mask][0])
# print("Replaced Array:", reflectance_corrected[mask][0])

In [ ]:
# fig, ax = plt.subplots(1,2, figsize=[20,14])

# img = ax[0].imshow(reflectance_rounded, cmap="Greys_r", vmin=0, vmax=0.3)

# fig.colorbar(img, orientation="vertical", pad=0.1, fraction=0.04)
# ax[0].set_title("Original Reflectance (Rounded)")

# img = ax[1].imshow((reflectance_corrected-reflectance_rounded)/reflectance_rounded*100, cmap="RdYlBu_r", vmin=-40, vmax=40)
# fig.colorbar(img, orientation="vertical", pad=0.1, fraction=0.04)
# ax[1].set_title("Correction Difference (%)")

# for ax0 in ax.ravel():
#   ax0.set(frame_on=True, xticks=[], yticks=[])

# plt.show()

In [ ]:
# Bins specified for floating point error corrections
# bins1 = np.unique(reflectance_rounded)
# bins2 = np.unique(reflectance_corrected)

In [ ]:
# create_density(reflectance_rounded, reflectance_corrected, label_corr="6S", units="Reflectance", bins=(bins1, bins2))